# Module 5 — group-safety repair: sync, rebuild, verify

Syncs six corrected files into `/content/drive/MyDrive/Fake_News_Detection`,
rebuilds the real X-FACT + FEVER corpus, then re-runs the read-only diagnosis.

**Run the cells in order.** Cell 3 syncs and runs the test suites. Cell 4 takes a
safety backup. Cell 5 rebuilds the real corpus. Cells 6-7 verify.

**Runtime: CPU is correct — no GPU is used.** If a GPU runtime is selected,
switch to CPU (Runtime > Change runtime type). It only makes allocation slower.

**If `drive.mount` fails with `ValueError: mount failed`:** this is a Colab/Drive
infrastructure timeout, not a problem with this notebook. Use
*Runtime > Disconnect and delete runtime*, reconnect, and run cell 1 again.
Do not add `force_remount=True` — forcing a remount tears down the existing
DriveFS binary and is a common cause of this exact failure.


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")          # no force_remount: it causes mount failures

PROJECT = Path("/content/drive/MyDrive/Fake_News_Detection")
print("project:", PROJECT, "exists:", PROJECT.exists())
assert PROJECT.exists(), (
    "Project folder not visible. Drive mounted but the path is wrong, "
    "or the shared folder is not added to My Drive.")

In [ ]:
PAYLOAD_JSON = r'''{"src/fact_verification/leakage_gate.py": {"b64": "...", "sha256": "3b300701ca656cd0b5c248ee260fd1a7f47e4f20be474d05047a8a30ddc5451b", "size": 21963}, "src/fact_verification/corpus_builder.py": {"b64": "...", "sha256": "9c9c3cbf16120ec85453cf56d577b772e261b4636542a1cf6179e6793f67331a", "size": 15308}, "tests/test_fact_verification.py": {"b64": "...", "sha256": "d98999f188ad085838a42b439b5d93546480f1c9be701e8915d173c6d82e805a", "size": 37512}, "scripts/build_claim_corpus.py": {"b64": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uCiIiIk1vZHVsZSA1IFN0YWdlIDMg4oCUIGJ1aWxkIHRoZSB1bmlmaWVkIGNsYWltL2V2aWRlbmNlIGNvcnB1cy4gQ09MQUIuCgpSZWFkcyB0aGUgZG93bmxvYWRlZCBYLUZBQ1QgYW5kIEZFVkVSIEpTT05MIGZpbGVzIGFuZCB3cml0ZXMgdGhlCmBjbGFpbXMuanNvbmxgIC8gYGV2aWRlbmNlLmpzb25sYCAvIGBjbGFpbV9zcGxpdHMuanNvbmAgYXJ0ZWZhY3RzIHRoZQpjb25maWcgYWxyZWFkeSBkZWNsYXJlcy4gTm8gcmV0cmlldmFsLCBubyBlbWJlZGRpbmdzLCBubyB0cmFpbmluZy4KCk9mZmljaWFsIHNwbGl0cyBhcmUgcHJlc2VydmVkIGV4YWN0bHkuIFRoZSBncm91cC1hd2FyZSA3MC8xNS8xNQpzcGxpdHRlciBpcyBOT1QgYXBwbGllZDogYm90aCBjb3Jwb3JhIHNoaXAgYSBwdWJsaXNoZWQgcHJvdG9jb2wsIGFuZApyZXNodWZmbGluZyBpdCB3b3VsZCBtYWtlIHJlc3VsdHMgaW5jb21wYXJhYmxlIHdpdGggcHJpb3Igd29yay4gVGhlCmxlYWthZ2UgZ2F0ZSBzdGlsbCBydW5zIGFnYWluc3QgdGhvc2Ugb2ZmaWNpYWwgc3BsaXRzIOKAlCBhIGZhaWx1cmUgdGhlcmUKaXMgYSBwcm9wZXJ0eSBvZiB0aGUgcHVibGlzaGVkIGRhdGFzZXQgYW5kIGlzIHJlcG9ydGVkIGFzIHN1Y2gsIG5ldmVyCnJlcGFpcmVkIGJ5IHNpbGVudGx5IHJlLXNwbGl0dGluZy4KClJlYWRzOiAgZGF0YXNldHMvbW9kdWxlNV9leHRlcm5hbC97eF9mYWN0LGZldmVyX2dvbGRfZXZpZGVuY2V9LyouanNvbmwKV3JpdGVzOiBkYXRhc2V0cy9mYWN0X3ZlcmlmaWNhdGlvbi97Y2xhaW1zLGV2aWRlbmNlfS5qc29ubCwKICAgICAgICBjbGFpbV9zcGxpdHMuanNvbiwgYW5kIG91dHB1dHMvbW9kdWxlNS9jb3JwdXNfYnVpbGRfcmVwb3J0Lntqc29uLG1kfQoKTmV2ZXIgdG91Y2hlcyBkYXRhc2V0cy9wcm9jZXNzZWQvLCBlbWJlZGRpbmdzLywgb3V0cHV0cy9tb2R1bGUzLywKb3V0cHV0cy9tb2R1bGU0LyBvciBtb2RlbHMvbW9kdWxlNC8uCgpVc2FnZToKICAgIHB5dGhvbiBzY3JpcHRzL2J1aWxkX2NsYWltX2NvcnB1cy5weQogICAgcHl0aG9uIHNjcmlwdHMvYnVpbGRfY2xhaW1fY29ycHVzLnB5IC0tb25seSB4X2ZhY3QKICAgIHB5dGhvbiBzY3JpcHRzL2J1aWxkX2NsYWltX2NvcnB1cy5weSAtLXNraXAtbGVha2FnZS1jaGVjayAgICMgbGFyZ2UgY29ycG9yYQoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQganNvbgppbXBvcnQgc3lzCmltcG9ydCB0aW1lCmZyb20gZGF0ZXRpbWUgaW1wb3J0IGRhdGV0aW1lLCB0aW1lem9uZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKUFJPSkVDVF9ST09UID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMV0Kc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihQUk9KRUNUX1JPT1QpKQoKZnJvbSBzcmMuZmFjdF92ZXJpZmljYXRpb24uY29ycHVzX2J1aWxkZXIgaW1wb3J0ICggICMgbm9xYTogRTQwMgogICAgYnVpbGRfY29ycHVzX2Zyb21fZmlsZXMsCiAgICBzcGxpdHNfZnJvbV9jbGFpbXMsCiAgICB3cml0ZV9jb3JwdXMsCikKZnJvbSBzcmMuZmFjdF92ZXJpZmljYXRpb24ubGVha2FnZV9nYXRlIGltcG9ydCAoICAjIG5vcWE6IEU0MDIKICAgIGVuZm9yY2VfZ3JvdXBfc2FmZV9zcGxpdHMsCiAgICBncm91cF9jbGFpbXMsCiAgICB2ZXJpZnlfbm9fbGVha2FnZSwKKQpmcm9tIHNyYy51dGlscy5jb25maWdfbG9hZGVyIGltcG9ydCBsb2FkX3lhbWwgICMgbm9xYTogRTQwMgpmcm9tIHNyYy51dGlscy5sb2dnZXIgaW1wb3J0IGdldF9sb2dnZXIgICMgbm9xYTogRTQwMgoKbG9nZ2VyID0gZ2V0X2xvZ2dlcigibW9kdWxlNS5idWlsZF9jb3JwdXMiKQoKRlJPWkVOID0gWyJkYXRhc2V0cy9wcm9jZXNzZWQiLCAiZW1iZWRkaW5ncyIsICJvdXRwdXRzL21vZHVsZTMiLCAib3V0cHV0cy9tb2R1bGU0IiwgIm1vZGVscy9tb2R1bGU0Il0KCiMgKGRpcmVjdG9yeSwgZGF0YXNldCB0YWcsIHtmaWxlbmFtZSBzdGVtIHN1ZmZpeCAtPiBvZmZpY2lhbCBzcGxpdCBuYW1lfSkKQ09SUFVTX0xBWU9VVCA9IHsKICAgICJ4X2ZhY3QiOiAoInhfZmFjdCIsICJ4X2ZhY3QiLCB7InRyYWluIjogInRyYWluIiwgImRldiI6ICJkZXYiLCAidGVzdCI6ICJ0ZXN0In0pLAogICAgImZldmVyIjogKCJmZXZlcl9nb2xkX2V2aWRlbmNlIiwgImZldmVyIiwgeyJ0cmFpbiI6ICJ0cmFpbiIsICJ2YWxpZGF0aW9uIjogInZhbGlkYXRpb24iLCAidGVzdCI6ICJ0ZXN0In0pLAp9CgoKZGVmIGZyb3plbl9zaWduYXR1cmUocm9vdDogUGF0aCkgLT4gZGljdFtzdHIsIEFueV06CiAgICBzaWc6IGRpY3Rbc3RyLCBBbnldID0ge30KICAgIGZvciByZWwgaW4gRlJPWkVOOgogICAgICAgIHAgPSByb290IC8gcmVsCiAgICAgICAgaWYgbm90IHAuZXhpc3RzKCk6CiAgICAgICAgICAgIHNpZ1tyZWxdID0gTm9uZQogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGZpbGVzID0gW2YgZm9yIGYgaW4gcC5yZ2xvYigiKiIpIGlmIGYuaXNfZmlsZSgpXQogICAgICAgIHNpZ1tyZWxdID0geyJuX2ZpbGVzIjogbGVuKGZpbGVzKSwgInRvdGFsX2J5dGVzIjogc3VtKGYuc3RhdCgpLnN0X3NpemUgZm9yIGYgaW4gZmlsZXMpfQogICAgcmV0dXJuIHNpZwoKCmRlZiBkaXNjb3Zlcl9maWxlcyhjb3JwdXNfZGlyOiBQYXRoLCBzcGxpdF9tYXA6IGRpY3Rbc3RyLCBzdHJdKSAtPiBsaXN0W3R1cGxlW1BhdGgsIHN0cl1dOgogICAgIiIiTWF0Y2ggZWFjaCBKU09OTCB0byBpdHMgb2ZmaWNpYWwgc3BsaXQgYnkgZmlsZW5hbWUgc3VmZml4LiIiIgogICAgZm91bmQ6IGxpc3RbdHVwbGVbUGF0aCwgc3RyXV0gPSBbXQogICAgZm9yIGZwIGluIHNvcnRlZChjb3JwdXNfZGlyLmdsb2IoIiouanNvbmwiKSk6CiAgICAgICAgc3RlbSA9IGZwLnN0ZW0ubG93ZXIoKQogICAgICAgIGZvciBzdWZmaXgsIG9mZmljaWFsIGluIHNwbGl0X21hcC5pdGVtcygpOgogICAgICAgICAgICBpZiBzdGVtLmVuZHN3aXRoKGYiX3tzdWZmaXh9Iik6CiAgICAgICAgICAgICAgICBmb3VuZC5hcHBlbmQoKGZwLCBvZmZpY2lhbCkpCiAgICAgICAgICAgICAgICBicmVhawogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGxvZ2dlci53YXJuaW5nKCJVbnJlY29nbmlzZWQgc3BsaXQgZm9yICVzIOKAlCBza2lwcGVkIChleHBlY3RlZCBzdWZmaXhlczogJXMpIiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgZnAubmFtZSwgc29ydGVkKHNwbGl0X21hcCkpCiAgICByZXR1cm4gZm91bmQKCgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249Ik1vZHVsZSA1IFN0YWdlIDM6IGJ1aWxkIHRoZSB1bmlmaWVkIGNsYWltIGNvcnB1cy4iKQogICAgcC5hZGRfYXJndW1lbnQoIi0tY29uZmlnIiwgZGVmYXVsdD1Ob25lKQogICAgcC5hZGRfYXJndW1lbnQoIi0tZGF0YS1kaXIiLCBkZWZhdWx0PSJkYXRhc2V0cy9tb2R1bGU1X2V4dGVybmFsIikKICAgIHAuYWRkX2FyZ3VtZW50KCItLW9ubHkiLCBjaG9pY2VzPVsieF9mYWN0IiwgImZldmVyIl0sIGRlZmF1bHQ9Tm9uZSkKICAgIHAuYWRkX2FyZ3VtZW50KCItLXNraXAtbGVha2FnZS1jaGVjayIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIsCiAgICAgICAgICAgICAgICAgICBoZWxwPSJTa2lwIHRoZSBjcm9zcy1zcGxpdCBuZWFyLWR1cGxpY2F0ZSBzY2FuIChleHBlbnNpdmUgb24gRkVWRVIpLiIpCiAgICBhcmdzID0gcC5wYXJzZV9hcmdzKCkKCiAgICByb290ID0gUFJPSkVDVF9ST09UCiAgICBjZmcgPSBsb2FkX3lhbWwoUGF0aChhcmdzLmNvbmZpZykgaWYgYXJncy5jb25maWcgZWxzZSByb290IC8gImNvbmZpZ3MiIC8gIm1vZHVsZTVfZmFjdF92ZXJpZmljYXRpb24ueWFtbCIpCiAgICBkYXRhX2RpciA9IHJvb3QgLyBhcmdzLmRhdGFfZGlyCiAgICBvdXRfcmVwb3J0cyA9IHJvb3QgLyBjZmdbIm91dHB1dCJdWyJyZXBvcnRzX2RpciJdCiAgICBvdXRfY29ycHVzX3Jvb3QgPSByb290IC8gY2ZnWyJvdXRwdXQiXVsiY29ycHVzX2RpciJdCiAgICBvdXRfcmVwb3J0cy5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCgogICAgYmVmb3JlID0gZnJvemVuX3NpZ25hdHVyZShyb290KQogICAgbGsgPSBjZmdbImxlYWthZ2UiXQoKICAgIHJlcG9ydDogZGljdFtzdHIsIEFueV0gPSB7CiAgICAgICAgIm1vZHVsZSI6ICJNb2R1bGUgNSBTdGFnZSAzIOKAlCBjbGFpbS9ldmlkZW5jZSBjb3JwdXMgY29uc3RydWN0aW9uIiwKICAgICAgICAiZ2VuZXJhdGVkX2F0IjogZGF0ZXRpbWUubm93KHRpbWV6b25lLnV0YykuaXNvZm9ybWF0KCksCiAgICAgICAgInNwbGl0X3BvbGljeSI6ICJQUkVTRVJWRV9PRkZJQ0lBTF9XSVRIX0dST1VQX1NBRkVUWV9SRVBBSVIiLAogICAgICAgICJzcGxpdF9wb2xpY3lfbm90ZSI6ICgiWC1GQUNUIGFuZCBGRVZFUiBib3RoIHNoaXAgb2ZmaWNpYWwgc3BsaXRzOyB0aGVzZSBhcmUgcHJlc2VydmVkICIKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgInZlcmJhdGltLiBUaGUgZ3JvdXAtYXdhcmUgNzAvMTUvMTUgc3BsaXR0ZXIgaXMgcmVzZXJ2ZWQgZm9yIGNvcnBvcmEgIgogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAid2l0aG91dCBhIHVzYWJsZSBvZmZpY2lhbCBzcGxpdCBhbmQgd2FzIE5PVCBhcHBsaWVkLiBXaGVyZSBhICIKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgImNvcnB1cydzIG93biBzcGxpdCBwbGFjZXMgbmVhci1kdXBsaWNhdGUgY2xhaW1zIG9uIGJvdGggc2lkZXMsIHRoZSAiCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJhZmZlY3RlZCBncm91cCBpcyBtb3ZlZCB3aG9sZSB0byBUUkFJTiAoZGV0ZXJtaW5pc3RpYywgbm8gUk5HLCBubyAiCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJkZWxldGlvbnMpIHNvIHRoYXQgb25lIGNsYWltX2dyb3VwX2lkIGJlbG9uZ3MgdG8gZXhhY3RseSBvbmUgIgogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAic3BsaXQuIEV2YWx1YXRpb24gc3BsaXRzIG9ubHkgZXZlciBzaHJpbmsuIG1ldGFkYXRhLm9mZmljaWFsX3NwbGl0ICIKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgInN0aWxsIGhvbGRzIGVhY2ggY29ycHVzJ3MgcHVibGlzaGVkIHNwbGl0OyBjbGFpbV9zcGxpdHMuanNvbiBpcyAiCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJ0aGUgYXV0aG9yaXRhdGl2ZSBmaW5hbCBhc3NpZ25tZW50LiBQZXItY2xhaW0gbW92ZXMgYXJlIGxpc3RlZCAiCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJ1bmRlciBncm91cF9zYWZldHlfcmVwYWlyLiIpLAogICAgICAgICJjb3Jwb3JhIjoge30sCiAgICAgICAgIndhcm5pbmdzIjogW10sCiAgICB9CgogICAgdGFyZ2V0cyA9IFthcmdzLm9ubHldIGlmIGFyZ3Mub25seSBlbHNlIFsieF9mYWN0IiwgImZldmVyIl0KICAgIGZvciBrZXkgaW4gdGFyZ2V0czoKICAgICAgICBzdWJkaXIsIGRhdGFzZXQsIHNwbGl0X21hcCA9IENPUlBVU19MQVlPVVRba2V5XQogICAgICAgIGNvcnB1c19kaXIgPSBkYXRhX2RpciAvIHN1YmRpcgogICAgICAgIGlmIG5vdCBjb3JwdXNfZGlyLmV4aXN0cygpOgogICAgICAgICAgICByZXBvcnRbIndhcm5pbmdzIl0uYXBwZW5kKGYie2RhdGFzZXR9OiB7Y29ycHVzX2Rpcn0gbm90IGZvdW5kIOKAlCBydW4gdGhlIGRvd25sb2FkIHNjcmlwdCBmaXJzdCIpCiAgICAgICAgICAgIGNvbnRpbnVlCgogICAgICAgIGZpbGVzID0gZGlzY292ZXJfZmlsZXMoY29ycHVzX2Rpciwgc3BsaXRfbWFwKQogICAgICAgIGlmIG5vdCBmaWxlczoKICAgICAgICAgICAgcmVwb3J0WyJ3YXJuaW5ncyJdLmFwcGVuZChmIntkYXRhc2V0fTogbm8gcmVjb2duaXNlZCBzcGxpdCBmaWxlcyBpbiB7Y29ycHVzX2Rpcn0iKQogICAgICAgICAgICBjb250aW51ZQoKICAgICAgICBsb2dnZXIuaW5mbygiWyVzXSBidWlsZGluZyBmcm9tICVkIGZpbGUocykuLi4iLCBkYXRhc2V0LCBsZW4oZmlsZXMpKQogICAgICAgIHQwID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgICAgIGNsYWltcywgZXZpZGVuY2VfcG9vbCwgc3RhdHMgPSBidWlsZF9jb3JwdXNfZnJvbV9maWxlcyhmaWxlcywgZGF0YXNldCwgY2ZnKQogICAgICAgIGJ1aWxkX3NlY29uZHMgPSByb3VuZCh0aW1lLnBlcmZfY291bnRlcigpIC0gdDAsIDIpCgogICAgICAgIGlmIG5vdCBzdGF0c1sicm93X2FjY291bnRpbmdfYmFsYW5jZXMiXToKICAgICAgICAgICAgcmVwb3J0WyJ3YXJuaW5ncyJdLmFwcGVuZCgKICAgICAgICAgICAgICAgIGYie2RhdGFzZXR9OiBST1cgQUNDT1VOVElORyBET0VTIE5PVCBCQUxBTkNFIOKAlCAiCiAgICAgICAgICAgICAgICBmIntzdGF0c1snbl9pbnB1dF9yb3dzJ119IGluIHZzIHtzdGF0c1snbl9jbGFpbXNfYnVpbHQnXX0gYnVpbHQgKyAiCiAgICAgICAgICAgICAgICBmIntzdGF0c1snbl9kcm9wcGVkX25vX2NsYWltX3RleHQnXX0gKyB7c3RhdHNbJ25fZHJvcHBlZF9iZWxvd19taW5fY2hhcnMnXX0gZHJvcHBlZCIpCgogICAgICAgIHNwbGl0cyA9IHNwbGl0c19mcm9tX2NsYWltcyhjbGFpbXMpCgogICAgICAgIGxvZ2dlci5pbmZvKCJbJXNdIGdyb3VwaW5nICVkIGNsYWltcyAodGhyZXNob2xkPSUuMmYsIG5ncmFtPSVkKS4uLiIsCiAgICAgICAgICAgICAgICAgICAgZGF0YXNldCwgbGVuKGNsYWltcyksIGxrWyJuZWFyX2R1cGxpY2F0ZV90aHJlc2hvbGQiXSwgbGtbIm5lYXJfZHVwbGljYXRlX25ncmFtIl0pCiAgICAgICAgdDEgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICAgICAgZ3JvdXBpbmcgPSBncm91cF9jbGFpbXMoY2xhaW1zLCBuZWFyX2R1cGxpY2F0ZT1UcnVlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG5ncmFtPWxrWyJuZWFyX2R1cGxpY2F0ZV9uZ3JhbSJdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHRocmVzaG9sZD1sa1sibmVhcl9kdXBsaWNhdGVfdGhyZXNob2xkIl0pCiAgICAgICAgZ3JvdXBfc2Vjb25kcyA9IHJvdW5kKHRpbWUucGVyZl9jb3VudGVyKCkgLSB0MSwgMikKCiAgICAgICAgIyBPZmZpY2lhbCBzcGxpdHMgYXJlIHByZXNlcnZlZCwgYnV0IGEgY29ycHVzIG1heSBpdHNlbGYgcGxhY2UgZHVwbGljYXRlCiAgICAgICAgIyBjbGFpbXMgb24gYm90aCBzaWRlcyBvZiBpdHMgb3duIHNwbGl0IGJvdW5kYXJ5IChGRVZFUiBkb2VzKS4gR3JvdXBzIGFyZQogICAgICAgICMgY29tcHV0ZWQgaW5kZXBlbmRlbnRseSBhYm92ZSwgc28gbm90aGluZyByZWNvbmNpbGVkIHRoZSB0d28gdW50aWwgaGVyZS4KICAgICAgICAjIFdob2xlIGdyb3VwcyBtb3ZlOyBubyBjbGFpbSBpcyBkZWxldGVkIGFuZCBubyBjb250ZW50IGNoYW5nZXMuCiAgICAgICAgc3BsaXRzLCByZXBhaXIgPSBlbmZvcmNlX2dyb3VwX3NhZmVfc3BsaXRzKGNsYWltcywgc3BsaXRzLCBncm91cGluZy5ncm91cF9pZHMpCiAgICAgICAgaWYgcmVwYWlyWyJuX2dyb3Vwc19yZXBhaXJlZCJdOgogICAgICAgICAgICBsb2dnZXIuaW5mbygiWyVzXSBncm91cC1zYWZldHkgcmVwYWlyOiAlZCBncm91cChzKSBzcGFubmVkIHNwbGl0cywgIgogICAgICAgICAgICAgICAgICAgICAgICAiJWQgY2xhaW0ocykgbW92ZWQ7IGNvdW50cyAlcyAtPiAlcyIsIGRhdGFzZXQsCiAgICAgICAgICAgICAgICAgICAgICAgIHJlcGFpclsibl9ncm91cHNfcmVwYWlyZWQiXSwgcmVwYWlyWyJuX2NsYWltc19tb3ZlZCJdLAogICAgICAgICAgICAgICAgICAgICAgICByZXBhaXJbImNvdW50c19iZWZvcmUiXSwgcmVwYWlyWyJjb3VudHNfYWZ0ZXIiXSkKICAgICAgICBlbHNlOgogICAgICAgICAgICBsb2dnZXIuaW5mbygiWyVzXSBncm91cC1zYWZldHkgcmVwYWlyOiBubyBncm91cCBzcGFubmVkIGEgc3BsaXQgKG5vLW9wKSIsIGRhdGFzZXQpCgogICAgICAgIGxlYWthZ2UgPSBOb25lCiAgICAgICAgaWYgbm90IGFyZ3Muc2tpcF9sZWFrYWdlX2NoZWNrOgogICAgICAgICAgICBsb2dnZXIuaW5mbygiWyVzXSBydW5uaW5nIGxlYWthZ2UgZ2F0ZSBvbiBPRkZJQ0lBTCBzcGxpdHMuLi4iLCBkYXRhc2V0KQogICAgICAgICAgICB0MiA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgICAgICAgICAgbGVha2FnZSA9IHZlcmlmeV9ub19sZWFrYWdlKAogICAgICAgICAgICAgICAgY2xhaW1zLCBzcGxpdHMsIGdyb3VwaW5nLmdyb3VwX2lkcywKICAgICAgICAgICAgICAgIG5lYXJfZHVwbGljYXRlX3RocmVzaG9sZD1sa1sibmVhcl9kdXBsaWNhdGVfdGhyZXNob2xkIl0sCiAgICAgICAgICAgICAgICBuZ3JhbT1sa1sibmVhcl9kdXBsaWNhdGVfbmdyYW0iXSkKICAgICAgICAgICAgbGVha2FnZVsic2Vjb25kcyJdID0gcm91bmQodGltZS5wZXJmX2NvdW50ZXIoKSAtIHQyLCAyKQogICAgICAgICAgICBsZWFrYWdlWyJpbnRlcnByZXRhdGlvbiJdID0gKAogICAgICAgICAgICAgICAgIk1lYXN1cmVkIG9uIHRoZSBjb3JwdXMncyBPV04gb2ZmaWNpYWwgc3BsaXRzLiBBIGZhaWx1cmUgaGVyZSBkZXNjcmliZXMgdGhlICIKICAgICAgICAgICAgICAgICJwdWJsaXNoZWQgZGF0YXNldCwgbm90IG91ciBzcGxpdHRpbmcuIFJlcG9ydCBpdDsgZG8gbm90IHNpbGVudGx5IHJlLXNwbGl0LiIpCiAgICAgICAgICAgIGlmIG5vdCBsZWFrYWdlWyJwYXNzZWQiXToKICAgICAgICAgICAgICAgIHJlcG9ydFsid2FybmluZ3MiXS5hcHBlbmQoCiAgICAgICAgICAgICAgICAgICAgZiJ7ZGF0YXNldH06IGxlYWthZ2UgZ2F0ZSBGQUlMRUQgb24gb2ZmaWNpYWwgc3BsaXRzIOKAlCBzZWUgY29ycHVzX2J1aWxkX3JlcG9ydC5qc29uIikKCiAgICAgICAgb3V0X2RpciA9IG91dF9jb3JwdXNfcm9vdCAvIGRhdGFzZXQKICAgICAgICBwYXRocyA9IHdyaXRlX2NvcnB1cyhjbGFpbXMsIGV2aWRlbmNlX3Bvb2wsIHNwbGl0cywgZ3JvdXBpbmcuZ3JvdXBfaWRzLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgIG91dF9kaXIsIGNmZ1sib3V0cHV0Il1bImNsYWltc19maWxlbmFtZSJdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgIHNwbGl0c19maWxlbmFtZT1jZmdbIm91dHB1dCJdWyJzcGxpdHNfZmlsZW5hbWUiXSkKCiAgICAgICAgcmVwb3J0WyJjb3Jwb3JhIl1bZGF0YXNldF0gPSB7CiAgICAgICAgICAgICJzb3VyY2VfZmlsZXMiOiBbZiJ7ZnAubmFtZX0gLT4ge3NwfSIgZm9yIGZwLCBzcCBpbiBmaWxlc10sCiAgICAgICAgICAgICJidWlsZF9zZWNvbmRzIjogYnVpbGRfc2Vjb25kcywKICAgICAgICAgICAgImdyb3VwaW5nX3NlY29uZHMiOiBncm91cF9zZWNvbmRzLAogICAgICAgICAgICAic3RhdGlzdGljcyI6IHN0YXRzLAogICAgICAgICAgICAic3BsaXRzIjoge2s6IGxlbih2KSBmb3IgaywgdiBpbiBzcGxpdHMuaXRlbXMoKX0sCiAgICAgICAgICAgICJncm91cGluZyI6IGdyb3VwaW5nLnN1bW1hcnksCiAgICAgICAgICAgICJncm91cF9zYWZldHlfcmVwYWlyIjogcmVwYWlyLAogICAgICAgICAgICAibGVha2FnZSI6IGxlYWthZ2UsCiAgICAgICAgICAgICJvdXRwdXRfZmlsZXMiOiBwYXRocywKICAgICAgICB9CiAgICAgICAgbG9nZ2VyLmluZm8oIlslc10gJWQgY2xhaW1zLCAlZCB1bmlxdWUgZXZpZGVuY2UgaXRlbXMsICVkIGdyb3VwcyAtPiAlcyIsCiAgICAgICAgICAgICAgICAgICAgZGF0YXNldCwgbGVuKGNsYWltcyksIGxlbihldmlkZW5jZV9wb29sKSwgZ3JvdXBpbmcubl9ncm91cHMsIG91dF9kaXIpCgogICAgYWZ0ZXIgPSBmcm96ZW5fc2lnbmF0dXJlKHJvb3QpCiAgICBjaGFuZ2VkID0ge2s6IHsiYmVmb3JlIjogYmVmb3JlLmdldChrKSwgImFmdGVyIjogYWZ0ZXIuZ2V0KGspfQogICAgICAgICAgICAgICBmb3IgayBpbiBhZnRlciBpZiBiZWZvcmUuZ2V0KGspICE9IGFmdGVyLmdldChrKX0KICAgIHJlcG9ydFsibW9kdWxlc18xXzRfaW50ZWdyaXR5Il0gPSB7InVuY2hhbmdlZCI6IG5vdCBjaGFuZ2VkLCAiY2hhbmdlZCI6IGNoYW5nZWR9CiAgICBpZiBjaGFuZ2VkOgogICAgICAgIHJlcG9ydFsid2FybmluZ3MiXS5hcHBlbmQoZiJGUk9aRU4gTU9EVUxFUyBDSEFOR0VEOiB7c29ydGVkKGNoYW5nZWQpfSIpCgogICAgKG91dF9yZXBvcnRzIC8gImNvcnB1c19idWlsZF9yZXBvcnQuanNvbiIpLndyaXRlX3RleHQoCiAgICAgICAganNvbi5kdW1wcyhyZXBvcnQsIGluZGVudD0yLCBlbnN1cmVfYXNjaWk9RmFsc2UsIGRlZmF1bHQ9c3RyKSwgZW5jb2Rpbmc9InV0Zi04IikKICAgIF93cml0ZV9tZChyZXBvcnQsIG91dF9yZXBvcnRzIC8gImNvcnB1c19idWlsZF9yZXBvcnQubWQiKQoKICAgIHByaW50KCJcbk1PRFVMRSA1IFNUQUdFIDMg4oCUIENMQUlNIENPUlBVUyBCVUlMRCIpCiAgICBwcmludCgiLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSIpCiAgICBmb3IgbmFtZSwgciBpbiByZXBvcnRbImNvcnBvcmEiXS5pdGVtcygpOgogICAgICAgIHMgPSByWyJzdGF0aXN0aWNzIl0KICAgICAgICBwcmludChmIlxue25hbWV9OiIpCiAgICAgICAgcHJpbnQoZiIgIGlucHV0IHJvd3MgICAgICAgIDoge3NbJ25faW5wdXRfcm93cyddfSIpCiAgICAgICAgcHJpbnQoZiIgIGNsYWltcyBidWlsdCAgICAgIDoge3NbJ25fY2xhaW1zX2J1aWx0J119ICAiCiAgICAgICAgICAgICAgZiIoZHJvcHBlZDoge3NbJ25fZHJvcHBlZF9ub19jbGFpbV90ZXh0J119IG5vLXRleHQsICIKICAgICAgICAgICAgICBmIntzWyduX2Ryb3BwZWRfYmVsb3dfbWluX2NoYXJzJ119IGJlbG93LW1pbikiKQogICAgICAgIHByaW50KGYiICByb3cgYWNjb3VudGluZyAgICA6IHsnQkFMQU5DRVMnIGlmIHNbJ3Jvd19hY2NvdW50aW5nX2JhbGFuY2VzJ10gZWxzZSAnTUlTTUFUQ0gnfSIpCiAgICAgICAgcHJpbnQoZiIgIG9mZmljaWFsIHNwbGl0cyAgIDoge3JbJ3NwbGl0cyddfSIpCiAgICAgICAgcHJpbnQoZiIgIGxhbmd1YWdlcyAgICAgICAgIDoge3NbJ2J5X2xhbmd1YWdlJ119IikKICAgICAgICBwcmludChmIiAgbGFiZWxzICAgICAgICAgICAgOiB7c1snYnlfbGFiZWwnXX0iKQogICAgICAgIHByaW50KGYiICBldmlkZW5jZSBpdGVtcyAgICA6IHtzWyduX3VuaXF1ZV9ldmlkZW5jZV9pdGVtcyddfSB1bmlxdWUgIgogICAgICAgICAgICAgIGYiKHtzWyduX2V2aWRlbmNlX21lbnRpb25zJ119IG1lbnRpb25zKSIpCiAgICAgICAgcHJpbnQoZiIgIGNsYWltcyB3L28gZXZpZGVuY2U6IHtzWyduX2NsYWltc193aXRoX25vX2V2aWRlbmNlJ119IikKICAgICAgICBwcmludChmIiAgZ3JvdXBzICAgICAgICAgICAgOiB7clsnZ3JvdXBpbmcnXVsnbl9ncm91cHMnXX0gIgogICAgICAgICAgICAgIGYiKGxhcmdlc3Qge3JbJ2dyb3VwaW5nJ11bJ2xhcmdlc3RfZ3JvdXBfc2l6ZSddfSwgIgogICAgICAgICAgICAgIGYie3JbJ2dyb3VwaW5nJ11bJ25lYXJfZHVwbGljYXRlX3BhaXJzJ119IG5lYXItZHVwIHBhaXJzKSIpCiAgICAgICAgaWYgclsibGVha2FnZSJdOgogICAgICAgICAgICBwcmludChmIiAgbGVha2FnZSBnYXRlICAgICAgOiB7J1BBU1MnIGlmIHJbJ2xlYWthZ2UnXVsncGFzc2VkJ10gZWxzZSAnRkFJTCd9ICIKICAgICAgICAgICAgICAgICAgZiIoe3JbJ2xlYWthZ2UnXVsnbl9wYXNzZWQnXX0ve3JbJ2xlYWthZ2UnXVsnbl9jaGVja3MnXX0sIHtyWydsZWFrYWdlJ11bJ3NlY29uZHMnXX1zKSIpCiAgICBwcmludChmIlxuTW9kdWxlcyAxLTQgdW5jaGFuZ2VkOiB7cmVwb3J0Wydtb2R1bGVzXzFfNF9pbnRlZ3JpdHknXVsndW5jaGFuZ2VkJ119IikKICAgIGlmIHJlcG9ydFsid2FybmluZ3MiXToKICAgICAgICBwcmludChmIlxuV0FSTklOR1MgKHtsZW4ocmVwb3J0Wyd3YXJuaW5ncyddKX0pOiIpCiAgICAgICAgZm9yIHcgaW4gcmVwb3J0WyJ3YXJuaW5ncyJdOgogICAgICAgICAgICBwcmludChmIiAgLSB7d30iKQogICAgcHJpbnQoZiJcbldyb3RlOiB7b3V0X3JlcG9ydHMvJ2NvcnB1c19idWlsZF9yZXBvcnQuanNvbid9IikKICAgIHByaW50KGYiV3JvdGU6IHtvdXRfcmVwb3J0cy8nY29ycHVzX2J1aWxkX3JlcG9ydC5tZCd9IikKCiAgICBmYWlsZWQgPSAobm90IHJlcG9ydFsibW9kdWxlc18xXzRfaW50ZWdyaXR5Il1bInVuY2hhbmdlZCJdCiAgICAgICAgICAgICAgb3IgYW55KG5vdCByWyJzdGF0aXN0aWNzIl1bInJvd19hY2NvdW50aW5nX2JhbGFuY2VzIl0gZm9yIHIgaW4gcmVwb3J0WyJjb3Jwb3JhIl0udmFsdWVzKCkpKQogICAgc3lzLmV4aXQoMSBpZiBmYWlsZWQgZWxzZSAwKQoKCmRlZiBfd3JpdGVfbWQocmVwb3J0OiBkaWN0W3N0ciwgQW55XSwgcGF0aDogUGF0aCkgLT4gTm9uZToKICAgIEwgPSBbIiMgTW9kdWxlIDUgU3RhZ2UgMyDigJQgQ2xhaW0vRXZpZGVuY2UgQ29ycHVzIEJ1aWxkIiwgIiIsCiAgICAgICAgIGYiR2VuZXJhdGVkOiB7cmVwb3J0WydnZW5lcmF0ZWRfYXQnXX0iLCAiIiwKICAgICAgICAgZiIqKlNwbGl0IHBvbGljeToge3JlcG9ydFsnc3BsaXRfcG9saWN5J119KioiLCAiIiwKICAgICAgICAgcmVwb3J0WyJzcGxpdF9wb2xpY3lfbm90ZSJdLCAiIl0KICAgIGZvciBuYW1lLCByIGluIHJlcG9ydFsiY29ycG9yYSJdLml0ZW1zKCk6CiAgICAgICAgcyA9IHJbInN0YXRpc3RpY3MiXQogICAgICAgIEwgKz0gW2YiIyMge25hbWV9IiwgIiIsCiAgICAgICAgICAgICAgInwgfCB8IiwgInwtLS18LS0tOnwiLAogICAgICAgICAgICAgIGYifCBJbnB1dCByb3dzIHwge3NbJ25faW5wdXRfcm93cyddfSB8IiwKICAgICAgICAgICAgICBmInwgQ2xhaW1zIGJ1aWx0IHwge3NbJ25fY2xhaW1zX2J1aWx0J119IHwiLAogICAgICAgICAgICAgIGYifCBEcm9wcGVkIChubyBjbGFpbSB0ZXh0KSB8IHtzWyduX2Ryb3BwZWRfbm9fY2xhaW1fdGV4dCddfSB8IiwKICAgICAgICAgICAgICBmInwgRHJvcHBlZCAoYmVsb3cgbWluIGNoYXJzKSB8IHtzWyduX2Ryb3BwZWRfYmVsb3dfbWluX2NoYXJzJ119IHwiLAogICAgICAgICAgICAgIGYifCBSb3cgYWNjb3VudGluZyBiYWxhbmNlcyB8IHtzWydyb3dfYWNjb3VudGluZ19iYWxhbmNlcyddfSB8IiwKICAgICAgICAgICAgICBmInwgVW5pcXVlIGV2aWRlbmNlIGl0ZW1zIHwge3NbJ25fdW5pcXVlX2V2aWRlbmNlX2l0ZW1zJ119IHwiLAogICAgICAgICAgICAgIGYifCBFdmlkZW5jZSBtZW50aW9ucyB8IHtzWyduX2V2aWRlbmNlX21lbnRpb25zJ119IHwiLAogICAgICAgICAgICAgIGYifCBDbGFpbXMgd2l0aG91dCBldmlkZW5jZSB8IHtzWyduX2NsYWltc193aXRoX25vX2V2aWRlbmNlJ119IHwiLAogICAgICAgICAgICAgIGYifCBDbGFpbSBncm91cHMgfCB7clsnZ3JvdXBpbmcnXVsnbl9ncm91cHMnXX0gfCIsCiAgICAgICAgICAgICAgZiJ8IExhcmdlc3QgZ3JvdXAgfCB7clsnZ3JvdXBpbmcnXVsnbGFyZ2VzdF9ncm91cF9zaXplJ119IHwiLCAiIiwKICAgICAgICAgICAgICBmIk9mZmljaWFsIHNwbGl0czogYHtyWydzcGxpdHMnXX1gIiwgIiIsCiAgICAgICAgICAgICAgZiJMYW5ndWFnZXM6IGB7c1snYnlfbGFuZ3VhZ2UnXX1gIiwgIiIsCiAgICAgICAgICAgICAgZiJMYWJlbHM6IGB7c1snYnlfbGFiZWwnXX1gIiwgIiIsCiAgICAgICAgICAgICAgZiJFdmlkZW5jZSBleHRyYWN0aW9uOiBge3NbJ2V2aWRlbmNlX2V4dHJhY3Rpb25fbWV0aG9kcyddfWAiLCAiIl0KICAgICAgICBpZiByWyJsZWFrYWdlIl06CiAgICAgICAgICAgIEwgKz0gW2YiIyMjIExlYWthZ2UgZ2F0ZSBvbiBvZmZpY2lhbCBzcGxpdHM6ICIKICAgICAgICAgICAgICAgICAgZiIqKnsnUEFTUycgaWYgclsnbGVha2FnZSddWydwYXNzZWQnXSBlbHNlICdGQUlMJ30qKiIsICIiLAogICAgICAgICAgICAgICAgICByWyJsZWFrYWdlIl1bImludGVycHJldGF0aW9uIl0sICIiXQogICAgICAgICAgICBmb3IgYyBpbiByWyJsZWFrYWdlIl1bImNoZWNrcyJdOgogICAgICAgICAgICAgICAgTC5hcHBlbmQoZiItIHsnT0snIGlmIGNbJ3Bhc3NlZCddIGVsc2UgJ0ZBSUwnfSBge2NbJ25hbWUnXX1gOiB7Y1snZGV0YWlsJ119IikKICAgICAgICAgICAgTC5hcHBlbmQoIiIpCiAgICBMICs9IFsiIyMgTW9kdWxlcyAxLTQgaW50ZWdyaXR5IiwgIiIsCiAgICAgICAgICBmIlVuY2hhbmdlZDogKip7cmVwb3J0Wydtb2R1bGVzXzFfNF9pbnRlZ3JpdHknXVsndW5jaGFuZ2VkJ119KioiLCAiIiwKICAgICAgICAgICIjIyBXYXJuaW5ncyIsICIiXSArIChbZiItIHt3fSIgZm9yIHcgaW4gcmVwb3J0WyJ3YXJuaW5ncyJdXSBvciBbIi0gbm9uZSJdKQogICAgcGF0aC53cml0ZV90ZXh0KCJcbiIuam9pbihMKSwgZW5jb2Rpbmc9InV0Zi04IikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==", "sha256": "416a845b16199ccb748fae32fce0e632868e2cba1d435af745c820d9985418a8", "size": 14260}, "scripts/audit_stage3_corpus.py": {"b64": "...", "sha256": "d75307f66206bad4b82b119640447fa260e56f2a8689c4e123c12824ab2913a4", "size": 20597}, "notebooks/module5_fever_leakage_diagnosis.py": {"b64": "...", "sha256": "3cf88c28d644b494123c639883ce4827df4cc33c5638f17e9d9f2ccf1e1ff5cd", "size": 17138}}'''
print('payload', len(PAYLOAD_JSON), 'chars')

In [ ]:
import base64, hashlib, json, re, shutil, subprocess, time
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Fake_News_Detection")
assert PROJECT.exists(), f"not found: {PROJECT}"
PAYLOAD = json.loads(PAYLOAD_JSON)

def run(cmd):
    x = subprocess.run(cmd, cwd=PROJECT, shell=True, capture_output=True, text=True)
    return x.returncode, x.stdout + x.stderr

print("DESTINATION CHECK")
plan = []
for rel, m in PAYLOAD.items():
    d = PROJECT / rel
    d.parent.mkdir(parents=True, exist_ok=True)
    act = "OVERWRITE" if d.exists() else "CREATE"
    plan.append((rel, act))
    print(f"  {act:9} {rel}  ({d.stat().st_size if d.exists() else 0} -> {m['size']} bytes)")

stamp = time.strftime("%Y%m%d_%H%M%S")
backup = PROJECT / "backups" / f"module5_group_safety_{stamp}"
n = 0
for rel, act in plan:
    if act == "OVERWRITE":
        b = backup / rel
        b.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(PROJECT / rel, b)
        n += 1
print(f"\nBacked up {n} source file(s) -> {backup if n else '(none needed)'}")

print("\nWRITE + HASH VERIFY")
for rel, m in PAYLOAD.items():
    d = PROJECT / rel
    d.write_bytes(base64.b64decode(m["b64"]))
    got = hashlib.sha256(d.read_bytes()).hexdigest()
    ok = got == m["sha256"]
    print(f"  {'OK  ' if ok else 'FAIL'} {rel}  {d.stat().st_size} bytes  {got[:16]}...")
    assert ok, f"HASH MISMATCH {rel}"

cleared = 0
for sub in ("src/fact_verification", "tests", "scripts"):
    p = PROJECT / sub / "__pycache__"
    if p.exists():
        shutil.rmtree(p, ignore_errors=True)
        cleared += 1
print(f"\nCleared {cleared} __pycache__ dir(s)")

rc, out = run("python -m py_compile src/fact_verification/leakage_gate.py "
              "src/fact_verification/corpus_builder.py scripts/build_claim_corpus.py "
              "scripts/audit_stage3_corpus.py")
print("\npy_compile:", "PASS" if rc == 0 else "FAIL " + out.strip()[:300])
assert rc == 0, "py_compile failed"

print("\nTEST SUITES")
failed = []
for t, exp in {"tests/test_fact_verification.py": 49,
               "tests/test_module5_corpus_schema.py": 21,
               "tests/test_contrastive.py": 22,
               "tests/test_embeddings.py": 5}.items():
    rc, out = run(f"python {t}")
    m = None
    for line in out.splitlines():
        h = re.search(r"(\d+)/(\d+) passed", line)
        if h:
            m = h
    ok = bool(m) and rc == 0 and int(m.group(1)) == int(m.group(2)) == exp
    print(f"  {'PASS' if ok else 'FAIL'}  {t}: {m.group(0) if m else out.strip()[-160:]}")
    if not ok:
        failed.append(t)
assert not failed, f"STOPPED: suites failed {failed}. Do not rebuild."

gate = (PROJECT / "src/fact_verification/leakage_gate.py").read_text(encoding="utf-8")
print("\ncontested-groups-to-train policy present:",
      "contested_groups_to_train" in gate)
print("\nSYNC COMPLETE. Corpus NOT yet rebuilt.")

In [ ]:
# Safety backup of the CURRENT FEVER corpus before it is overwritten.
# Optional but recommended: the rebuild replaces claims.jsonl / evidence.jsonl /
# claim_splits.json in place.
import zipfile
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Fake_News_Detection")
src = PROJECT / "datasets/fact_verification/fever"
zip_path = PROJECT / "backups" / "fever_corpus_before_group_safety.zip"
zip_path.parent.mkdir(parents=True, exist_ok=True)

names = ["claims.jsonl", "evidence.jsonl", "claim_splits.json"]
missing = [n for n in names if not (src / n).is_file()]
if missing:
    print("SKIPPED — no existing corpus to back up:", missing)
else:
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for n in names:
            p = src / n
            print(f"  adding {n} ({p.stat().st_size:,} bytes)")
            z.write(p, arcname=n)
    print(f"\nbackup: {zip_path}  ({zip_path.stat().st_size:,} bytes)")

In [ ]:
# Rebuild on LOCAL disk, then copy results to Drive.
# Building directly against the Drive FUSE mount is slow, and that long
# wall-clock time on an idle GPU is what tripped Colab's usage limits.
import hashlib, shutil, subprocess, sys, time
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Fake_News_Detection")
WORK = Path("/content/m5_build")

print("Staging code + inputs on local disk...")
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)
t0 = time.perf_counter()
for sub in ("src", "scripts", "configs", "tests"):
    shutil.copytree(PROJECT / sub, WORK / sub,
                    ignore=shutil.ignore_patterns("__pycache__"))
shutil.copytree(PROJECT / "datasets/module5_external", WORK / "datasets/module5_external")
staged = sum(f.stat().st_size for f in WORK.rglob("*") if f.is_file())
print("  staged %.1f MB in %.1fs -> %s" % (staged / 1e6, time.perf_counter() - t0, WORK))

print()
print("REBUILDING (X-FACT + FEVER). No --skip-leakage-check, no --only.")
print()
t1 = time.perf_counter()
proc = subprocess.Popen([sys.executable, "scripts/build_claim_corpus.py"], cwd=WORK,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
log = []
for line in proc.stdout:
    print(line, end="", flush=True)      # keeps the tab visibly active
    log.append(line)
proc.wait()
print()
print("exit code: %s   build wall-clock: %.1fs" % (proc.returncode, time.perf_counter() - t1))
assert proc.returncode == 0, "BUILD FAILED - stop and report the log above"

print()
print("Copying results back to Drive...")
for rel in ("datasets/fact_verification", "outputs/module5"):
    src, dst = WORK / rel, PROJECT / rel
    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print("  %s: %d file(s)" % (rel, sum(1 for f in src.rglob("*") if f.is_file())))
(PROJECT / "outputs/module5").mkdir(parents=True, exist_ok=True)
(PROJECT / "outputs/module5/rebuild_console.log").write_text("".join(log), encoding="utf-8")

# Module 1-4 integrity, checked directly against Drive. The staged tree has no
# datasets/processed, so the builder's own frozen-signature check is a no-op here.
print()
print("Module 1-4 integrity (read-only, against Drive):")
FROZEN = {"datasets/processed/train.csv": "fc26454e56cb9704509ffe3216d52a22",
          "datasets/processed/validation.csv": "081f7b828477ab9f27725d17c34b525a",
          "datasets/processed/test.csv": "66599a4f9d71bd5cee1927c3a573f0ce"}
bad = []
for rel, exp in FROZEN.items():
    f = PROJECT / rel
    if not f.is_file():
        print("  MISSING " + rel)
        bad.append(rel)
        continue
    h = hashlib.md5()
    with f.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    ok = h.hexdigest() == exp
    print("  %s  %s  %s" % ("PASS" if ok else "FAIL", rel, h.hexdigest()))
    if not ok:
        bad.append(rel)
assert not bad, "Module 1-4 changed: %s" % bad
print()
print("Rebuild complete and copied to Drive.")

In [ ]:
import json
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Fake_News_Detection")
rep = json.loads((PROJECT / "outputs/module5/corpus_build_report.json").read_text(encoding="utf-8"))
print(f"split_policy: {rep['split_policy']}\n")
for ds, c in rep["corpora"].items():
    rp, lk = c["group_safety_repair"], c["leakage"]
    print(f"=== {ds} ===")
    print(f"  policy          : {rp['policy']}  target={rp.get('target_split')}")
    print(f"  groups repaired : {rp['n_groups_repaired']}")
    print(f"  claims moved    : {rp['n_claims_moved']}   deleted: {rp['claims_deleted']}")
    print(f"  counts before   : {rp['counts_before']}")
    print(f"  counts after    : {rp['counts_after']}   delta {rp['delta']}")
    print(f"  leakage gate    : passed={lk['passed']} {lk['n_passed']}/{lk['n_checks']}")
    for ch in lk["checks"]:
        if not ch["passed"]:
            print(f"    FAILED {ch['name']}: {ch['detail'][:150]}")
    for m in rp["moves"][:25]:
        print(f"    {m['claim_id']}  group={m['claim_group_id']}  "
              f"{m['from_split']} -> {m['to_split']}  members={m['member_counts']}")
    print()

In [ ]:
import subprocess, sys
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Fake_News_Detection")
r = subprocess.run([sys.executable, "notebooks/module5_fever_leakage_diagnosis.py"],
                   cwd=PROJECT, capture_output=True, text=True)
print(r.stdout[-14000:])
if r.returncode:
    print(r.stderr[-2000:])
print("diagnosis exit:", r.returncode)